<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/DistilBERT_IMDB_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# In this example we will once again do an end to end fine tuning of a sentiment classifier using DistilBERT with IMDB Data as base model.


In [1]:
# Install dependencies

%pip install -q --upgrade transformers datasets evaluate accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 288.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 67.0 MB/s eta 0:00:00


In [9]:
# Perform the imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np
import warnings
from datasets import load_dataset
import collections
warnings.filterwarnings('ignore')


In [4]:
# Let us verify if GPU or Apple Silicon is present or not

if torch.cuda.is_available():
  print(f'✅ GPU is available')
elif torch.backends.mps.is_available():
  print(f'✅ Apple Silicon is available')
else:
  print("⚠️ No GPU detected. Fine-tuning will be very slow.")
  print("   In Colab: Runtime → Change runtime type → GPU")


✅ GPU is available


In [7]:
# Now let us load the dataset

dataset = load_dataset('stanfordnlp/imdb')

#explore the dataset
print(f'==== IMDB ====')
print(f'Splits: {list(dataset.keys())}')
print(f'Training Examples: {len(dataset["train"])}')
print(f'Test Examples: {len(dataset["test"])}')
print(f'Training Features: {dataset["train"].features}')

==== IMDB ====
Splits: ['train', 'test', 'unsupervised']
Training Examples: 25000
Test Examples: 25000
Training Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [12]:
# Now let us do basic sanity check on data like label distribution
label_counts = collections.Counter(dataset["train"]["label"])
print(f'Label Counts: {dict(label_counts)}')
print(f'0: Negative; 1: Positive')

# Print one Sample Review
print(f'=== Sample Review ===')
print(f'Text: {dataset["train"][0]["text"][:200]}...')
print(f'Label: {dataset["train"][0]["label"]} ({'Positive' if dataset["train"][0]["label"] == "1" else 'Negative'})')

Label Counts: {0: 12500, 1: 12500}
0: Negative; 1: Positive
=== Sample Review ===
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
Label: 0 (Negative)


In [13]:
# Now that we have loaded the dataset, we need to tokenize the full dataset

CHECK_POINT = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(CHECK_POINT)
tokenized = dataset.map(lambda x: tokenizer(x["text"], truncation=True, max_length=256))
print(f'Tokenization Complete')
print(f'Columns: {tokenized["train"].column_names}')
print(f'Sample input_ids lehgth: {len(tokenized["train"][0]["input_ids"])}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenization Complete
Columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Sample input_ids lehgth: 256


In [22]:
# Now that data is tokenized, we will pad the data with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [14]:
#Before we run the full training let us first run training on a small subset to see if everyting is working fine because we don't want to waste compute

small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
small_test = tokenized["test"].shuffle(seed=42).select(range(1000))

print(f'Small Training Subset: {len(small_train)} examples')
print(f'Small Test Subset: {len(small_test)} examples')

Small Training Subset: 2000 examples
Small Test Subset: 1000 examples


In [15]:
# Now let us load the model which will be fine tuned.

model = AutoModelForSequenceClassification.from_pretrained(CHECK_POINT, num_labels=2)
print(f'Parameters: {sum(p.numel() for p in model.parameters())}')


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 66955010


In [24]:
# Model is loaded, before we start fine tuning it, we need to define evaluation metrics

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  acc = accuracy_metric.compute(predictions=predictions, references=labels)
  f1 = f1_metric.compute(predictions=predictions, references=labels)
  return {"f1": f1["f1"], "accuracy": acc["accuracy"]}

print(f'✅ Metrics Evaluation Function Defined')

✅ Metrics Evaluation Function Defined


In [25]:
# Now that evaluation function is defined, let us configure the training pipeline

training_args = TrainingArguments(
    output_dir='output/novapay-sentiment',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

print(f'✅ Training Arguments Configured')
print(f'Epochs: {training_args.num_train_epochs}')
print(f'Batch Size: {training_args.per_device_train_batch_size}')
print(f'Learning Rate: {training_args.learning_rate}')

✅ Training Arguments Configured
Epochs: 2
Batch Size: 16
Learning Rate: 2e-05


In [26]:
# now let us train

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()
print(f'✅ Training Complete')
print(f'Training Loss: {train_result.training_loss:.4f}')


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.248468,0.388331,0.866732,0.865000
2,0.156153,0.405482,0.874510,0.872000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Complete
Training Loss: 0.2046


In [28]:
# Now that training is done (small data) let us see the evaluation results

eval_results = trainer.evaluate()
print("=== Evaluation Results ===")
print(f'Accuracy: {eval_results['eval_accuracy']:.4f}')
print(f'F1: {eval_results['eval_f1']:.4f}')
print(f'Eval Loss: {eval_results['eval_loss']:.4f}')



Training Loss,Validation Loss,Epoch,F1,Accuracy
0.156153,0.405482,2,0.874510,0.872000


=== Evaluation Results ===
Accuracy: 0.8720
F1: 0.8745
Eval Loss: 0.4055


In [29]:
# Now that small Training is working and we can see Evaluation results let's run the big training

full_trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=tokenized['train'].shuffle(seed=42),
    eval_dataset=tokenized['test'].shuffle(seed=42),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

full_train_result = full_trainer.train()
print(f'✅ Full Training Complete')
print(f'Training Loss: {full_train_result.training_loss:.4f}')


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.178379,0.303338,0.902956,0.897040
2,0.198699,0.281857,0.915417,0.914240


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Full Training Complete
Training Loss: 0.2155


In [30]:
# Priht the full trainig evaluation results

full_eval_results = full_trainer.evaluate()
print("=== Full Evaluation Results ===")
print(f'Accuracy: {full_eval_results['eval_accuracy']:.4f}')
print(f'F1: {full_eval_results['eval_f1']:.4f}')
print(f'Eval Loss: {full_eval_results['eval_loss']:.4f}')


Training Loss,Validation Loss,Epoch,F1,Accuracy
0.198699,0.281857,2,0.915417,0.914240


=== Full Evaluation Results ===
Accuracy: 0.9142
F1: 0.9154
Eval Loss: 0.2819


In [31]:
# Now let us push the fine tuned model to the Hugging Face Hub

full_trainer.push_to_hub('novapay-sentiment-distilbert')

print(f'✅ Model Pushed to HuggingFace Hub')
print(f'Model URL: https://huggingface.co/abhishes/novapay-sentiment-distilbert')



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model Pushed to HuggingFace Hub
Model URL: https://huggingface.co/abhishes/novapay-sentiment-distilbert


In [34]:
# Create Model Card Content. This needs to be updated on Hugging Face

model_card_content = """
---
  license: apache-2.0
  tags:
    - text-classification
    - sentiment-analysis
    - nova-pay
  datasets:
    - imdb
  metrics:
    - accuracy
    - f1
---
# Novapay Sentiment Classifier (DistilBERT)

## Intended Use
Fine Tuned for sentiment classification of NovaPay Customer Support Messages.
This is a proof-of-concept model trained on IMDB Dataset

## Training Data
- **DataSet**: IMDB
- **BaseModel**: distilbert/distilbert-base-uncased
- **Epochs**: 2
- **Learning Rate: **  2e-5

## Evaluation Results
- **Accuracy: ** 0.9142
- **F1 Score: ** 0.9154

##Limitations
- Trained on IMDB Movie Review not on actual customer support messages
- May not generalize well to financial support domain
- POC Only Not for produciton use

"""

print(model_card_content)


---
  license: apache-2.0
  tags: 
    - text-classification
    - sentiment-analysis
    - nova-pay
  datasets:
    - imdb
  metrics:
    - accuracy
    - f1
---
# Novapay Sentiment Classifier (DistilBERT)

## Intended Use 
Fine Tuned for sentiment classification of NovaPay Customer Support Messages.
This is a proof-of-concept model trained on IMDB Dataset

## Training Data
- **DataSet**: IMDB
- **BaseModel**: distilbert/distilbert-base-uncased
- **Epochs**: 2
- **Learning Rate: **  2e-5

## Evaluation Results
- **Accuracy: ** 0.9142
- **F1 Score: ** 0.9154

##Limitations
- Trained on IMDB Movie Review not on actual customer support messages
- May not generalize well to financial support domain 
- POC Only Not for produciton use


